# Project 2: Market-Leverage Factor
Factor: **LEV = high-minus-low market leverage**, `debt / (debt + market equity)`, sorted monthly at the 30th/70th percentiles, returns Jan 1974 - Dec 2018.

Pipeline: (1) Compustat debt, (2) CRSP returns, (3) merge + timing, (4) factor, regressions, sensitivity, bad times.

**What this does:** imports the libraries. `pandas`/`numpy` for data, `io`/`zipfile`/`urllib` only to download the Fama-French files later (regressions are written by hand in numpy).

In [1]:
import pandas as pd, numpy as np, io, zipfile, urllib.request

## 1. Compustat: quarterly debt

**What this does:** loads the raw Compustat file (one row per company x quarter x CRSP link) and builds one clean row per company-quarter with total debt.
- Keeps only the fundamentals columns, then drops repeats caused by a company having several CRSP links, then true duplicate company-quarters.
- Drops financial firms (SIC 6000-6999): their debt is part of their business, so leverage means something different.
- Total debt = `dlcq` (debt in current liabilities) + `dlttq` (long-term debt). A missing piece counts as 0, but a firm must report at least one of them.
- The printed table is the "how many observations each step affects" log required by the assignment.

In [2]:
raw = pd.read_csv("compustat.csv.gz", dtype={"GVKEY": str, "cusip": str, "sic": str},
                  parse_dates=["datadate", "rdq", "LINKDT"])
raw.columns = raw.columns.str.lower()
raw["linkenddt"] = pd.to_datetime(raw["linkenddt"].replace("E", pd.NaT), errors="coerce")

log = [("raw rows (one per company-quarter-link)", len(raw))]

fund_cols = ["gvkey", "datadate", "fyearq", "fqtr", "rdq", "conm", "tic", "cusip",
             "sic", "costat", "atq", "ceqq", "seqq", "dlcq", "dlttq"]
fund = raw[fund_cols].drop_duplicates()
log.append(("remove repeats caused by multiple links", len(fund)))

fund = fund.drop_duplicates(["gvkey", "datadate"], keep="first")
log.append(("remove duplicate company-quarters", len(fund)))

fund = fund[fund["datadate"] <= "2025-12-31"]
log.append(("trim to 2025", len(fund)))

sic = pd.to_numeric(fund["sic"], errors="coerce")
fund = fund[~sic.between(6000, 6999)]
log.append(("drop financials (SIC 6000-6999)", len(fund)))

fund = fund[fund[["dlcq", "dlttq"]].notna().any(axis=1)]
log.append(("has debt data", len(fund)))

# total debt = short-term + long-term (a missing piece counts as 0)
fund["debt"] = fund["dlcq"].fillna(0) + fund["dlttq"].fillna(0)
fund = fund[fund["debt"] >= 0]
log.append(("non-negative debt", len(fund)))

out = pd.DataFrame(log, columns=["step", "n_obs"]); out["dropped"] = -out["n_obs"].diff()
print(out)

                                      step    n_obs   dropped
0  raw rows (one per company-quarter-link)  1215146       NaN
1  remove repeats caused by multiple links  1203381   11765.0
2        remove duplicate company-quarters  1202753     628.0
3                             trim to 2025  1191721   11032.0
4          drop financials (SIC 6000-6999)   895530  296191.0
5                            has debt data   862299   33231.0
6                        non-negative debt   862296       3.0


**What this does:** builds the **link table** (which CRSP stock/PERMNO belongs to which Compustat company, and over what dates) and saves the PERMNO list that was used to pull CRSP.
It is needed because one company can have several share classes (several PERMNOs) and we want leverage measured at the *company* level.

In [3]:
# link table: which PERMNOs belong to which company (needed to sum market equity across share classes)
links = (raw[["gvkey", "lpermno", "linkprim", "linktype", "linkdt", "linkenddt"]].drop_duplicates())
links["linkenddt"] = links["linkenddt"].fillna(pd.Timestamp("2099-12-31"))
links = links[links["gvkey"].isin(fund["gvkey"].unique())]

links["lpermno"].astype(int).drop_duplicates().sort_values().to_csv("permnos.txt", index=False, header=False)
print(links["gvkey"].nunique(), "companies,", links["lpermno"].nunique(), "PERMNOs")

21120 companies, 21639 PERMNOs


> **Data note:** the share of firms with zero reported debt falls from ~19% to ~6% in 2019Q1 (a Compustat reporting break), so the sample ends in 2018. Some pre-2019 zeros may be missing values coded as 0.

## 2. CRSP: monthly total returns
`MthRet` is CRSP's total return (includes dividends, adjusted for splits and distributions). Frequency: monthly. Market equity = |price| x shares outstanding / 1000 ($ millions).

**What this does:** loads and cleans CRSP monthly stock data.
- Keeps US common stock only (`EQTY`/`COM`/`NS`, US-incorporated), listed on NYSE/AMEX/NASDAQ, 1973-2025.
- Market equity `me` = |price| x shares outstanding / 1000, in $ millions (CRSP prices can be negative to flag bid/ask averages, hence `abs`). Shares are in thousands; Compustat debt is in millions, so units now match.
- Removes exact duplicate rows (every duplicate permno-month was an identical row) and asserts none remain.
- Returns are CRSP `MthRet`: a total return including dividends and adjusted for splits/distributions, so no manual price adjustment is needed.

In [4]:
crsp = pd.read_csv("crsp_monthly.csv.gz", dtype={"HdrCUSIP": str})
crsp.columns = crsp.columns.str.lower()
crsp["mthcaldt"] = pd.to_datetime(crsp["mthcaldt"])

log = [("raw CRSP rows", len(crsp))]

crsp = crsp[(crsp["securitytype"] == "EQTY") & (crsp["securitysubtype"] == "COM") &
            (crsp["sharetype"] == "NS") & (crsp["usincflg"] == "Y")]
log.append(("US common stock", len(crsp)))

crsp = crsp[crsp["primaryexch"].isin(["N", "A", "Q"])]
log.append(("NYSE/AMEX/NASDAQ", len(crsp)))

crsp = crsp[(crsp["mthcaldt"] >= "1973-01-01") & (crsp["mthcaldt"] <= "2025-12-31")]
log.append(("1973-2025", len(crsp)))

crsp["me"] = crsp["mthprc"].abs() * crsp["shrout"] / 1000
crsp = crsp[crsp["me"] > 0]
log.append(("positive market equity", len(crsp)))

# all duplicate permno-months are exact duplicate rows
crsp = crsp.drop_duplicates().sort_values(["permno", "mthcaldt"])
assert not crsp.duplicated(["permno", "mthcaldt"]).any()
log.append(("remove exact duplicate rows", len(crsp)))

out = pd.DataFrame(log, columns=["step", "n_obs"]); out["dropped"] = -out["n_obs"].diff()
print(out)
print("missing MthRet:", crsp["mthret"].isna().sum())


                          step    n_obs   dropped
0                raw CRSP rows  2961391       NaN
1              US common stock  2569656  391735.0
2             NYSE/AMEX/NASDAQ  2518587   51069.0
3                    1973-2025  2518587      -0.0
4       positive market equity  2508393   10194.0
5  remove exact duplicate rows  2500417    7976.0
missing MthRet: 869


## 3. Merge: firm market equity, leverage, and timing
Fundamentals are matched to month t only after their public date (`rdq`, else `datadate` + 3 months), so leverage at month t uses no future information. The stock's return in month t+1 is the return we explain.

**What this does:** turns the debt data into a monthly **market leverage** measure per company.
1. Attaches each stock to its company using only links valid in that month, then sums market equity across all share classes of the company (`firm_me`).
2. Works out when each quarter's numbers became public: the earnings-announcement date `rdq`, else quarter-end + 3 months. This prevents look-ahead bias.
3. `merge_asof` gives each company-month the latest debt figure already public (at most 365 days old).
4. `mkt_lev = debt / (debt + market equity)`. `book_lev = debt / total assets` is kept for the sensitivity test (set to missing if assets <= 0).

In [5]:
# stock -> company link valid in that month; firm ME sums all share classes
m = crsp[["permno", "mthcaldt", "me"]].merge(
        links[["gvkey", "lpermno", "linkprim", "linkdt", "linkenddt"]],
        left_on="permno", right_on="lpermno", how="inner")
m = m[(m["mthcaldt"] >= m["linkdt"]) & (m["mthcaldt"] <= m["linkenddt"])]
m = m.sort_values(["permno", "mthcaldt", "linkprim"]).drop_duplicates(["permno", "mthcaldt"])

firm_me = (m.groupby(["gvkey", "mthcaldt"], as_index=False)["me"].sum()
             .rename(columns={"me": "firm_me"}).sort_values("mthcaldt"))
print(len(m), "stock-months matched;", len(firm_me), "company-months")

fund["avail"] = fund["rdq"].where(fund["rdq"] >= fund["datadate"])
fund["avail"] = fund["avail"].fillna(fund["datadate"] + pd.DateOffset(months=3))
fund = fund.sort_values("avail")

panel = pd.merge_asof(firm_me, fund[["gvkey", "avail", "datadate", "debt", "atq"]],
                      left_on="mthcaldt", right_on="avail", by="gvkey",
                      direction="backward", tolerance=pd.Timedelta(days=365))
panel = panel.dropna(subset=["debt"])
panel["mkt_lev"]  = panel["debt"] / (panel["debt"] + panel["firm_me"])
panel["book_lev"] = panel["debt"] / panel["atq"].where(panel["atq"] > 0)     # for the sensitivity test
print(len(panel), "company-months with leverage")
print(panel[["mkt_lev", "book_lev"]].describe(percentiles=[.01, .5, .99]).round(3))

2468294 stock-months matched; 2442175 company-months
2264320 company-months with leverage
           mkt_lev     book_lev
count  2264320.000  2222492.000
mean         0.249        0.249
std          0.246        1.063
min          0.000        0.000
1%           0.000        0.000
50%          0.177        0.213
99%          0.906        0.955
max          1.000      855.111


**What this does:** builds the final stock-month dataset used for the factor.
- Each row is a stock with its company's leverage at the end of month *t*, and its **next month's return** `ret_next` (the return in month t+1). Sorting on information known at t and earning the t+1 return is what makes the test forward-looking.
- `ret_next` is only kept if the next row really is the next calendar month (no gaps).
- Keeps return months Jan 1974 - Dec 2018 and saves `stock_timed.csv`.

In [6]:
# stock-month panel: each PERMNO gets its company's leverage and NEXT month's return
stock = (m[["permno", "mthcaldt", "gvkey", "me"]]
         .merge(panel[["gvkey", "mthcaldt", "mkt_lev", "book_lev"]], on=["gvkey", "mthcaldt"], how="inner")
         .merge(crsp[["permno", "mthcaldt", "mthret"]], on=["permno", "mthcaldt"], how="left"))
stock["ym"] = stock["mthcaldt"].dt.to_period("M")
stock = stock.sort_values(["permno", "ym"])

stock["next_ym"]  = stock.groupby("permno")["ym"].shift(-1)
stock["ret_next"] = stock.groupby("permno")["mthret"].shift(-1)
stock.loc[stock["next_ym"] != stock["ym"] + 1, "ret_next"] = np.nan      # only if next row is next calendar month
stock["ret_month"] = stock["ym"] + 1

log = [("stock-months with leverage", len(stock))]
stock = stock[stock["ret_next"].notna()]
log.append(("has next-month return", len(stock)))
stock = stock[(stock["ret_month"] >= "1974-01") & (stock["ret_month"] <= "2018-12")]
log.append(("returns Jan 1974 - Dec 2018", len(stock)))

out = pd.DataFrame(log, columns=["step", "n_obs"]); out["dropped"] = -out["n_obs"].diff()
print(out)
stock.to_csv("stock_timed.csv", index=False)

                          step    n_obs   dropped
0   stock-months with leverage  2290286       NaN
1        has next-month return  2270209   20077.0
2  returns Jan 1974 - Dec 2018  2005744  264465.0


## 4. Factor construction and winsorizing

**What this does:** loads the stock-month panel and **winsorizes**.
- Next-month returns are clipped at the 1st/99th percentile *within each month*, so a handful of extreme returns (e.g. +1,900%) cannot drive a portfolio. The cell prints how many observations were changed.
- Market leverage is not winsorized: it lives in [0,1] and is only used to rank stocks, so clipping would not change any portfolio. Book leverage has huge outliers (up to 855), so its top 1% is clipped.

In [7]:
st = pd.read_csv("stock_timed.csv", usecols=["permno","gvkey","ret_month","me","mkt_lev","book_lev","ret_next"])
st["ret_month"] = pd.PeriodIndex(st["ret_month"], freq="M")
print("obs:", len(st), "| stocks:", st.permno.nunique(), "| months:", st.ret_month.nunique())

def winsorize(df, col, lo=.01, hi=.99):
    g = df.groupby("ret_month")[col]
    a, b = g.transform(lambda s: s.quantile(lo)), g.transform(lambda s: s.quantile(hi))
    clipped = df[col].clip(a, b)
    print(f"{col}: {(clipped != df[col]).sum():,} obs changed ({lo:.0%}/{hi:.0%} by month)")
    return clipped

st["ret_w"]  = winsorize(st, "ret_next")
st["book_w"] = winsorize(st, "book_lev", 0, .99)   # book leverage has outliers up to 855; mkt_lev is bounded in [0,1] and only ranked

obs: 2005744 | stocks: 16801 | months: 540
ret_next: 40,272 obs changed (1%/99% by month)
book_lev: 54,318 obs changed (0%/99% by month)


**What this does:** defines the three building blocks, reused for every factor variant.
- `assign`: each month, finds the leverage breakpoints (e.g. 30th/70th percentile) and labels every stock 0 (low) to k (high).
- `port_ret`: portfolio return each month, value-weighted by market equity (or equal-weighted).
- `lev_factor`: the factor itself, **LEV = return of the high-leverage group minus the low-leverage group**, a zero-cost long-short portfolio.

In [8]:
def assign(df, col, cuts):
    """Bucket 0..k each month by breakpoints on `col` (cuts e.g. [.3,.7])."""
    bp = df.groupby("ret_month")[col].quantile(cuts).unstack()
    bp.columns = range(len(cuts))
    x = df[["ret_month", col]].join(bp, on="ret_month")
    return sum((x[col] > x[i]).astype(int) for i in bp.columns)

def port_ret(df, by, weight="vw", ret="ret_w"):
    df = df.assign(w=df["me"] if weight == "vw" else 1.0)
    df["wr"] = df["w"] * df[ret]
    g = df.groupby(["ret_month"] + by)[["wr", "w"]].sum()
    return (g["wr"] / g["w"]).unstack(by)

def lev_factor(df, col="mkt_lev", cuts=(.3,.7), weight="vw", ret="ret_w"):
    """High-minus-Low leverage."""
    df = df.assign(b=assign(df, col, list(cuts)))
    p = port_ret(df, ["b"], weight, ret)
    return (p[len(cuts)] - p[0]).rename("LEV")

## 5. Fama-French + momentum factors (Ken French data library)

**What this does:** downloads the benchmark factors from Ken French's data library: market excess return (`Mkt-RF`), size (`SMB`), value (`HML`), the risk-free rate (`RF`) and momentum (`UMD`). Converts percent to decimals and trims to Jan 1974 - Dec 2018 to match our sample.

In [9]:
def french(name, skip):
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{name}_CSV.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    df = pd.read_csv(io.StringIO(z.read(z.namelist()[0]).decode()), skiprows=skip, skipinitialspace=True)
    df = df.rename(columns={df.columns[0]: "ym"})
    df = df[df["ym"].astype(str).str.fullmatch(r"\d{6}")]
    df.index = pd.PeriodIndex(pd.to_datetime(df.pop("ym").astype(str), format="%Y%m"), freq="M")
    return df.astype(float) / 100

ff  = french("F-F_Research_Data_Factors", 3)
mom = french("F-F_Momentum_Factor", 13).rename(columns=lambda c: c.strip()).rename(columns={"Mom": "UMD"})
ff  = ff.join(mom[["UMD"]]).loc["1974-01":"2018-12"]
print(ff.describe().round(4).loc[["count","mean","std"]])

         Mkt-RF       SMB       HML        RF      UMD
count  540.0000  540.0000  540.0000  540.0000  540.000
mean     0.0059    0.0021    0.0031    0.0038    0.006
std      0.0450    0.0300    0.0288    0.0029    0.043


## 6. Regressions (OLS, Newey-West t-stats with 6 lags)

**What this does:** a hand-written OLS regression returning the intercept (alpha), coefficients, t-stats, R-squared and N.
The t-stats use **Newey-West standard errors (6 lags)**, which correct for autocorrelation and heteroskedasticity in monthly returns. Written in numpy because `statsmodels` does not import in this environment.

In [10]:
def ols(y, X, lags=6):
    d = pd.concat([y, X], axis=1).dropna()
    yy, XX = d.iloc[:, 0].values, np.column_stack([np.ones(len(d)), d.iloc[:, 1:].values])
    b = np.linalg.solve(XX.T @ XX, XX.T @ yy)
    e = yy - XX @ b
    S = (XX * e[:, None]).T @ (XX * e[:, None])
    for l in range(1, lags + 1):
        G = (XX[l:] * e[l:, None]).T @ (XX[:-l] * e[:-l, None])
        S += (1 - l / (lags + 1)) * (G + G.T)
    A = np.linalg.inv(XX.T @ XX)
    se = np.sqrt(np.diag(A @ S @ A))
    names = ["alpha"] + list(X.columns)
    return pd.Series({**dict(zip(names, b)), **{f"t_{n}": v for n, v in zip(names, b / se)},
                      "R2": 1 - e.var() / yy.var(), "N": len(d)})

**What this does:** the main factor tests (requirements 5 and 6).
- Builds LEV and reports its average monthly return and t-stat: is there a premium?
- Regresses LEV on CAPM, FF3 and FF3+momentum. If the **alpha** is ~0, existing factors already explain LEV and it is not a new source of return.

In [11]:
lev = lev_factor(st)
d = ff.join(lev, how="inner")
print(f"LEV: mean {lev.mean()*100:.2f}%/mo, t = {ols(lev, pd.DataFrame(index=lev.index))['t_alpha']:.2f}")

models = {"CAPM": ["Mkt-RF"], "FF3": ["Mkt-RF","SMB","HML"], "FF3+UMD": ["Mkt-RF","SMB","HML","UMD"],
          "CAPM+LEV": ["Mkt-RF","LEV"], "FF3+UMD+LEV": ["Mkt-RF","SMB","HML","UMD","LEV"]}

print("\nIs LEV spanned by existing factors?")
for mname in ["CAPM","FF3","FF3+UMD"]:
    r = ols(d["LEV"], d[models[mname]]); print(f"  {mname:8s} alpha {r['alpha']*100:5.2f}%  t {r['t_alpha']:5.2f}  R2 {r['R2']:.2f}")

LEV: mean 0.07%/mo, t = 0.36

Is LEV spanned by existing factors?
  CAPM     alpha  0.23%  t  1.09  R2 0.08
  FF3      alpha -0.22%  t -1.84  R2 0.60
  FF3+UMD  alpha -0.04%  t -0.39  R2 0.64


**What this does:** the test-asset regressions.
- Forms 25 portfolios (5 size x 5 leverage groups), value-weighted, and converts to excess returns.
- Regresses each on five models: CAPM, FF3, FF4 (FF3+UMD), CAPM+LEV, FF4+LEV.
- Comparing R-squared with and without LEV shows whether LEV adds explanatory power. `b_LEV`/`t_LEV` are its loading and t-stat in the full model, and `alpha` is what is left unexplained.
- Caveat: these portfolios are *sorted* on leverage, so LEV partly explains them by construction.

In [12]:
# test assets: 25 size x leverage portfolios (excess returns)
st["size_q"] = assign(st, "me", [.2,.4,.6,.8]); st["lev_q"] = assign(st, "mkt_lev", [.2,.4,.6,.8])
ta = port_ret(st, ["size_q","lev_q"]); ta.columns = [f"S{a}L{b}" for a, b in ta.columns]
ta = ta.sub(ff["RF"], axis=0).loc[d.index]

rows = []
for p in ta.columns:
    row = {"port": p}
    for mname, f in models.items():
        r = ols(ta[p], d[f]); row[f"R2_{mname}"] = r["R2"]
        if mname == "FF3+UMD+LEV": row.update(b_LEV=r["LEV"], t_LEV=r["t_LEV"], alpha=r["alpha"], t_alpha=r["t_alpha"])
    rows.append(row)
res = pd.DataFrame(rows).set_index("port")
print(res.round(3).to_string())
print("Avg R2:", res.filter(like="R2_").mean().round(3).to_dict())

      R2_CAPM  R2_FF3  R2_FF3+UMD  R2_CAPM+LEV  R2_FF3+UMD+LEV  b_LEV   t_LEV  alpha  t_alpha
port                                                                                         
S0L0    0.375   0.624       0.665        0.376           0.667 -0.147  -1.782  0.001    0.619
S0L1    0.422   0.648       0.676        0.423           0.679 -0.167  -1.882 -0.003   -1.351
S0L2    0.421   0.651       0.685        0.421           0.686 -0.086  -1.074 -0.002   -1.158
S0L3    0.418   0.622       0.663        0.430           0.664  0.058   0.735 -0.003   -1.558
S0L4    0.398   0.623       0.681        0.434           0.682  0.078   0.778 -0.002   -1.065
S1L0    0.560   0.799       0.814        0.567           0.817 -0.158  -3.056  0.001    0.349
S1L1    0.562   0.808       0.819        0.568           0.821 -0.148  -3.339 -0.004   -2.386
S1L2    0.573   0.820       0.841        0.574           0.841 -0.021  -0.510 -0.001   -0.905
S1L3    0.576   0.825       0.846        0.615           0.8

## 7. Sensitivity

**What this does:** the robustness table (requirement 7). It re-runs the factor test (mean return and FF4 alpha) under alternative assumptions:
- how the factor is formed: quintiles vs terciles, equal- vs value-weighted, book vs market leverage;
- winsorizing: none vs 1/99;
- sample: 1974-95 vs 1996-2018, and excluding the 2007-09 crisis.

In [13]:
def summarize(f, label, ffx=ff):
    d = ffx.join(f.rename("LEV"), how="inner").dropna()
    r = ols(d["LEV"], d[["Mkt-RF","SMB","HML","UMD"]])
    return {"spec": label, "mean%": d["LEV"].mean()*100,
            "t_mean": ols(d["LEV"], pd.DataFrame(index=d.index))["t_alpha"],
            "FF4 alpha%": r["alpha"]*100, "t_alpha": r["t_alpha"], "R2": r["R2"], "N": len(d)}

specs = [("Base: 30/70, VW, mkt lev", lev),
         ("Quintiles 20/80",           lev_factor(st, cuts=(.2,.8))),
         ("Terciles 33/67",            lev_factor(st, cuts=(1/3,2/3))),
         ("Equal-weighted legs",       lev_factor(st, weight="ew")),
         ("Book leverage",             lev_factor(st, col="book_w")),
         ("No winsorizing of returns", lev_factor(st, ret="ret_next"))]
rows = [summarize(f, s) for s, f in specs]
rows += [summarize(lev, "1974-1995", ff.loc[:"1995-12"]), summarize(lev, "1996-2018", ff.loc["1996-01":]),
         summarize(lev, "Excl. 2007-09 crisis", ff[~ff.index.to_series().between(pd.Period("2007-12"), pd.Period("2009-06"))])]
print(pd.DataFrame(rows).set_index("spec").round(3).to_string())

                           mean%  t_mean  FF4 alpha%  t_alpha     R2    N
spec                                                                     
Base: 30/70, VW, mkt lev   0.074   0.360      -0.044   -0.394  0.639  540
Quintiles 20/80            0.129   0.527      -0.062   -0.463  0.624  540
Terciles 33/67             0.064   0.350      -0.029   -0.285  0.649  540
Equal-weighted legs        0.188   1.001       0.081    0.784  0.648  540
Book leverage             -0.156  -0.954      -0.283   -2.512  0.487  540
No winsorizing of returns  0.075   0.366      -0.043   -0.390  0.638  540
1974-1995                  0.241   1.083      -0.029   -0.203  0.697  264
1996-2018                 -0.086  -0.256      -0.009   -0.055  0.667  276
Excl. 2007-09 crisis       0.080   0.381      -0.061   -0.528  0.646  521


## 8. Bad times: rational vs behavioral

**What this does:** the bad-times evidence (requirement 8). Takes LEV's average return in NBER recessions, in months the market fell, in the worst 10% of market months, and in expansions.
A *risk* premium should come with poor payoffs in bad times; a factor that does well in bad times behaves like a hedge.

In [14]:
rec = [("1974-01","1975-03"),("1980-01","1980-07"),("1981-07","1982-11"),
       ("1990-07","1991-03"),("2001-03","2001-11"),("2007-12","2009-06")]   # NBER recessions
d = ff.join(lev, how="inner"); d["recession"] = False
for a, b in rec: d.loc[a:b, "recession"] = True
d["mkt_down"]  = d["Mkt-RF"] < 0
d["mkt_worst"] = d["Mkt-RF"] <= d["Mkt-RF"].quantile(.10)

out = {k: d.loc[d[k], "LEV"].agg(["mean","count"]) for k in ["recession","mkt_down","mkt_worst"]}
out["expansion"] = d.loc[~d["recession"], "LEV"].agg(["mean","count"])
print(pd.DataFrame(out).T.assign(mean=lambda x: x["mean"]*100).round(2))

           mean  count
recession  0.02   76.0
mkt_down   1.39  215.0
mkt_worst  2.77   54.0
expansion  0.08  464.0


## 9. Results and interpretation

**Setup.** 2,005,744 stock-months, 16,801 stocks, 540 months (Jan 1974 - Dec 2018). LEV = value-weighted returns of the top 30% minus the bottom 30% of stocks by market leverage, sorted monthly. Returns are CRSP total returns, winsorized at 1/99 (about 40,000 observations changed).

**1. LEV does not earn a premium.** Average return is +0.07% per month (about 0.9% a year) with a monthly standard deviation of 4.3%; the Newey-West t-stat is 0.36. It is not statistically different from zero.

**2. LEV is mostly a value factor in disguise.**

| Benchmark | LEV alpha (per month) | t | R-squared |
|---|---|---|---|
| CAPM | +0.23% | 1.09 | 0.08 |
| FF3 | -0.22% | -1.84 | 0.60 |
| FF3 + UMD | -0.04% | -0.39 | 0.64 |

In the four-factor regression LEV loads **+1.02 on HML (t = 13.3)**, -0.21 on UMD (t = -4.1), -0.10 on the market (t = -3.2) and is not related to SMB. This makes sense mechanically: market leverage has market equity in its denominator, so high-leverage stocks tend to have low market value and high book-to-market. After controlling for the standard factors the alpha is essentially zero, so LEV adds no new return.

**3. LEV adds only modest explanatory power for the 25 size x leverage portfolios.** Average R-squared: CAPM 0.631, CAPM+LEV 0.675, FF3 0.819, FF4 0.842, **FF4+LEV 0.853**. LEV is significant (|t| > 2) in 18 of the 25 portfolios, but the average gain over FF4 is only about 1.1 percentage points (largest: 8 points, for the biggest, lowest-leverage portfolio, 0.81 to 0.89). Only 3 of 25 alphas are significant in the full model. This should be read with care, because the test portfolios are sorted on leverage, so some explanatory power is mechanical.

**4. The result is not robust, and the premium is never significant.**

| Specification | Mean %/mo (t) | FF4 alpha %/mo (t) |
|---|---|---|
| Base (30/70, VW, market leverage) | 0.07 (0.36) | -0.04 (-0.39) |
| Quintiles 20/80 | 0.13 (0.53) | -0.06 (-0.46) |
| Terciles | 0.06 (0.35) | -0.03 (-0.29) |
| Equal-weighted legs | 0.19 (1.00) | +0.08 (0.78) |
| Book leverage | -0.16 (-0.95) | **-0.28 (-2.51)** |
| No winsorizing | 0.08 (0.37) | -0.04 (-0.39) |
| 1974-1995 | 0.24 (1.08) | -0.03 (-0.20) |
| 1996-2018 | -0.09 (-0.26) | -0.01 (-0.06) |
| Excluding 2007-09 | 0.08 (0.38) | -0.06 (-0.53) |

The premium changes sign across subperiods and weighting schemes. The one significant number (book leverage, FF4 alpha -0.28%) is a single result out of nine specifications, so it could easily be chance.

**5. Bad times: LEV behaves like a hedge, not a risk premium.**

| Market state | LEV mean return (per month) | Months |
|---|---|---|
| NBER recessions | +0.02% | 76 |
| Months the market fell | +1.39% | 215 |
| Worst 10% market months | +2.77% | 54 |
| Expansions | +0.08% | 464 |

A rational (risk-based) story says leveraged firms should earn a premium because they do badly in bad times. Here LEV does *well* when the market falls (negative market beta of -0.26 in CAPM), and it earns nothing in recessions. That is the opposite of compensation for bad-times risk.

**Interpretation.**
- *Rational vs behavioral:* there is no premium to explain. The hedge-like pattern does not support the distress-risk story, and nothing here shows a behavioral mispricing either, because LEV's return is explained by value and momentum exposure.
- *Market efficiency:* leverage is computed from public accounting and price data, so this tests **semi-strong** efficiency. A zero abnormal return after controlling for known factors is consistent with an efficient market: investors cannot use public leverage information to earn extra return.
- *For the audience (e.g. a quant fund considering a leverage tilt):* a leverage tilt gives the same exposure as a value tilt, which they already have, with no extra return and a premium that is unstable across periods. After trading costs there is no reason to add it as a separate factor.

**Caveats.** (a) Compustat debt before 2019 may contain missing values coded as 0, which would mis-classify some firms as zero-leverage; the sample ends in 2018 for this reason. (b) Leverage uses current market equity, so it partly proxies for past price moves (value and momentum), as the HML and UMD loadings show. (c) The 25 test portfolios are sorted on leverage, which flatters LEV's explanatory power. (d) These are 9 specifications, so one significant result is not strong evidence.